In [ ]:
import pandas as pd

In [ ]:
date_to_load_from = "2026-10-24 00:00:00"
df = pd.read_csv("orders_20261024.csv")

Clean data from csv creating the orders fact table
deduplicate, flag and the invalid orders, and lowercase the order status

In [ ]:
# get the rows I want for the fact table "Orders"
orders = df[["order_id","customer_id", "order_date", "product", "total_amount", "status","country", "updated_at"]]
#make sure the rows are from after the cutoffs
cutoff_time = pd.to_datetime(date_to_load_from)
orders["updated_at"] = pd.to_datetime(orders["updated_at"])
orders = orders[orders["updated_at"] > cutoff_time]
#drop duplicated and keep the last entry
orders = orders.drop_duplicates(subset="order_id", keep="last")
flagged_orders = orders[orders["total_amount"].isna()]
#valid orders are te ones with a valid total amount for order... can change based on business needs
valid_orders = orders[orders["total_amount"].notna()]
#lowercase the status' to make them more uniform for querying later on
valid_orders['status'] = valid_orders['status'].str.lower()
#clean total_amount by stripping the $ 
valid_orders['total_amount'] = valid_orders['total_amount'].str.replace('$','',regex=False).astype(float)
valid_orders


df for cutomer dimesion table

In [ ]:
customers = df[["customer_id", "customer_name", "email","country"]]

dimesion table for products

In [212]:
products = df[["product","category","unit_price"]]
products["unit_price"] = products["unit_price"].str.replace('$','').astype(float)
products = products.drop_duplicates(subset="product")
products['category'] = products['category'].str.title()
products['product'] = products['product'].str.title()
products

,product,category,unit_price
0,Yoga Mat,Outdoors,29.90
1,Green Tea Box,Groceries,7.80
2,Water Bottle,Outdoors,15.00
3,Usb-C Cable,Electronics,9.99
5,Office Chair,Furniture,189.50
7,Wireless Mouse,Electronics,24.99
10,Coffee Beans 1Kg,Grocery,18.20
11,Standing Desk,Furniture,349.00
16,Gel Pens,Stationery,6.49
29,Notebook 3-Pack,Stationery,12.75


make the database connection

In [ ]:
import os
import re

from psycopg.pq import ExecStatus
import pandas as pd
from sqlalchemy import text
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
load_dotenv()

    # Fix the connection string to use the modern async psycopg driver
db_url = re.sub(r'^postgresql:', 'postgresql+psycopg:', os.getenv('DATABASE_URL', ''))
engine = create_engine(db_url, echo=True)

# 2. Open a clean connection block to run a quick test query
with engine.connect() as connection:
    result = connection.execute(text("SELECT 'Connection Successful!'"))
    print(result.scalar())

# 3. Read directly into Pandas (Works perfectly without errors!)
df = pd.read_sql("SELECT * FROM orders LIMIT 5;", con=engine)
print(df)

put dataframe data in database and count the rows in staging orders table and count rows in df and then perform a merge automatically if counts match up... if not alert for further inspection

In [ ]:
import pandas as pd
from sqlalchemy import text

valid_orders.to_sql("orders_temp", con=engine, if_exists="replace", index=False)
query = "SELECT count(order_id) from orders_temp"
df = pd.read_sql(query, con=engine)
local_count = len(valid_orders)

# FIX: Added [0, 0] to extract the primitive integer value from the dataframe
db_count = int(df.iloc[0, 0]) 
print(db_count)

if local_count == db_count:
    merge_query = text("""
    MERGE INTO orders AS target
    USING orders_temp AS source
    ON target.order_id = source.order_id::text
    WHEN MATCHED THEN
        UPDATE SET 
            total_amount = source.total_amount::double precision, 
            status = source.status, 
            updated_at = source.updated_at::timestamp,
            order_date = source.order_date::date,
            product = source.product,
            country =  source.country
    WHEN NOT MATCHED THEN
        INSERT (order_id, customer_id, order_date, product,  status, country, total_amount, updated_at)
        VALUES (
            source.order_id::text, 
            source.customer_id::integer,
            source.order_date::date,
            source.product,
            source.status, 
            source.country,
            source.total_amount::double precision, 
            source.updated_at::timestamp
        );
""") 
else:
    print("ALERT")
    print("mismatch with data")
    merge_query = None

if merge_query is not None:
    with engine.begin() as connection:
        connection.execute(merge_query)
        connection.execute(text("drop table orders_temp;"))


Insert Products into a temp table

In [217]:
products.to_sql(name='products_temp',
    con=engine,
    if_exists='replace',
    index=False
    )

2026-10-26 20:05:09,357 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-26 20:05:09,359 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s::VARCHAR AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s::VARCHAR, %(param_2)s::VARCHAR, %(param_3)s::VARCHAR, %(param_4)s::VARCHAR, %(param_5)s::VARCHAR]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s::VARCHAR
2026-10-26 20:05:09,359 INFO sqlalchemy.engine.Engine [cached since 1.235e+05s ago] {'table_name': 'products_temp', 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-10-26 20:05:09,486 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog

-1

merge to prod products table

In [215]:
sql_products_temp_count_query = "SELECT count(product) from products_temp"
products_df = pd.read_sql(sql_products_temp_count_query, con=engine)
local_products_count = len(products)
db_products_count = int(products_df.iloc[0,0])
print(local_products_count)
print(db_products_count)

if local_products_count == db_products_count:
    products_merge = text("""
        Merge into products as target
        Using products_temp as source 
        On target.product = source.product
        When Matched Then 
        Update Set
            category = source.category,
            unit_price = source.unit_price
        WHEN NOT MATCHED THEN
        Insert(product, category, unit_price)
        VALUES (source.product, source.category, source.unit_price)
        """
    )
else:
    print("Mismatch with product counts")
    products_merge = None

if products_merge is not None:
    with engine.begin() as connection:
        connection.execute(products_merge)
        connection.execute(text("DROP table products_temp"))


2026-10-26 20:04:25,987 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-26 20:04:25,988 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s::VARCHAR AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s::VARCHAR, %(param_2)s::VARCHAR, %(param_3)s::VARCHAR, %(param_4)s::VARCHAR, %(param_5)s::VARCHAR]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s::VARCHAR
2026-10-26 20:04:25,989 INFO sqlalchemy.engine.Engine [cached since 1.235e+05s ago] {'table_name': 'SELECT count(product) from products_temp', 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-10-26 20:04:26,114 INFO sqlalchemy.engine.Engine SELECT count(product) from products_temp
2026-10-26 20:04:26,115 INFO sqlalc